# Registration

Registration estimates the **rigid transform between two scans**: a source cloud `pos` and a reference cloud `ref_pos`, and the $4 \times 4$ matrix $T$ with $q = R p + t$ that moves the source into the reference frame. A model returns point correspondences and `decode` fits the transform to them.

![A 3DMatch pair before and after GeoTransformer aligns it](../assets/tasks/registration.png)


## Create a model

`create_model` builds the architecture and loads its weights; `return_info=True` also gives the registered transform, input keys and measured metrics.

In [ ]:
import torch_pointcloud as tp

model, info = tp.create_model(
    "geotransformer.3dmatch.zheng-qin",
    task="registration",
    pretrained=True,
    return_info=True,
)
model = model.cuda().eval()

print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
print(f"Input keys: {info['input_keys']}")
print(f"Metrics: {info['weights']['metrics']}")
print(tp.list_models(task="registration", pretrained=True))

# Parameters: 9.83M
# Input keys: ('x', 'pos', 'batch', 'ref_x', 'ref_pos', 'batch_ref_pos')
# Metrics: {'RR': 92.48, 'FMR': 98.14, 'IR': 70.93, '3DLoMatch/RR': 74.36, '3DLoMatch/FMR': 87.07, '3DLoMatch/IR': 43.54}
# ['geotransformer.3dmatch.zheng-qin', 'geotransformer.kitti.zheng-qin', 'geotransformer.modelnet40.zheng-qin']

The GeoTransformer checkpoints read a constant feature, so `x` and `ref_x` are `None` and the model fills them with ones.

| Argument        | Shape           | Description                                                        |
| --------------- | --------------- | ------------------------------------------------------------------ |
| `x`             | $(N, C)$        | Source features, or `None`                                         |
| `pos`           | $(N, 3)$        | Source points                                                      |
| `batch`         | $(N,)$          | Pair index of each source point                                    |
| `ref_x`         | $(M, C)$        | Reference features, or `None`                                      |
| `ref_pos`       | $(M, 3)$        | Reference points                                                   |
| `batch_ref_pos` | $(M,)$          | Pair index of each reference point                                 |
| **returns**     | dict of tensors | Superpoint and patch matching scores, which `model.decode(...)` turns into a transform |

`decode` returns a `Registration3D`: `transform` $(B, 4, 4)$ and the packed correspondences `src_points`, `ref_points`, `scores` and `batch`.

## Load the dataset

[`ThreeDMatch`](../api/datasets/threedmatch.md) holds the pairs of the 3DMatch and 3DLoMatch benchmarks. Pass `download=True` the first time: the fragments come from the PREDATOR release and the pair lists from the GeoTransformer repository.

The reference cloud is a ragged per-pair tensor, so it collates as a `cat_keys` entry and arrives with `batch_ref_pos`; the ground-truth transform is one matrix per pair, a `stack_keys` entry.

In [ ]:
import torch

from torch_pointcloud.datasets import ThreeDMatch
from torch_pointcloud.metrics import inlier_ratio, rotation_error, translation_error
from torch_pointcloud.utils.data import DataKeys, collate

dataset = ThreeDMatch(root="data", split="test", benchmark="3DMatch", download=True)
sample = dataset[0]
print(f"Pairs: {len(dataset)}")
print({k: tuple(v.shape) if hasattr(v, "shape") else v for k, v in sample.items()})

data = collate([sample], cat_keys=[DataKeys.REF_POS], stack_keys=[DataKeys.TRANSFORM])
data = {k: v.cuda() if torch.is_tensor(v) else v for k, v in data.items()}

with torch.no_grad():
    out = model(None, data["pos"], data["batch"], None, data["ref_pos"], data["batch_ref_pos"])
registration = model.decode(out)
print(f"Point correspondences: {registration['src_points'].shape[0]}")
print(registration["transform"][0].cpu().numpy().round(4))

transform, target = registration["transform"], data["transform"]
print(f"RRE {rotation_error(transform, target).item():.3f} deg, RTE {100 * translation_error(transform, target).item():.2f} cm")
print(f"Inlier ratio: {inlier_ratio(registration['src_points'], registration['ref_points'], target, 0.1, registration['batch']).item():.3f}")

# Pairs: 1623
# {'pos': (19082, 3), 'ref_pos': (18977, 3), 'transform': (4, 4), 'overlap': (), 'frame': '1', 'ref_frame': '0', 'scene': '7-scenes-redkitchen', 'information': (6, 6)}
# Point correspondences: 6037
# [[ 0.9976  0.0551 -0.0417 -0.1208]
#  [-0.0544  0.9983  0.0185 -0.0396]
#  [ 0.0427 -0.0162  0.999   0.1177]
#  [ 0.      0.      0.      1.    ]]
# RRE 0.716 deg, RTE 0.60 cm
# Inlier ratio: 0.884

`transform_points` moves the source into the reference frame with the estimated transform:

In [ ]:
from torch_pointcloud.ops.geometry import transform_points

aligned = transform_points(data["pos"], registration["transform"][0])
print(aligned.shape)

# torch.Size([19082, 3])

## Benchmark

The metrics live in `torch_pointcloud.metrics` and score a batch of pairs at once: `inlier_ratio` on the correspondences (IR), `root_mean_square_alignment_error`, the 3DMatch alignment error read through the information matrix of the pair, and the recalls built on them, `feature_matching_recall` (FMR, inlier ratio above 5%) and `registration_recall` (RR, RMSE under 0.2 m).

In [ ]:
from torch.utils.data import Subset
from tqdm.auto import tqdm

from torch_pointcloud.metrics import feature_matching_recall, registration_recall, root_mean_square_alignment_error
from torch_pointcloud.utils.data import PointCloudDataLoader

dataloader = PointCloudDataLoader(
    Subset(dataset, range(50)),
    batch_size=1,
    cat_keys=[DataKeys.REF_POS],
    stack_keys=[DataKeys.TRANSFORM, DataKeys.INFORMATION],
)

ratios, rmses = [], []
with torch.no_grad():
    for data in tqdm(dataloader, desc="Evaluating"):
        data = {k: v.cuda() if torch.is_tensor(v) else v for k, v in data.items()}
        out = model(None, data["pos"], data["batch"], None, data["ref_pos"], data["batch_ref_pos"])
        registration = model.decode(out)
        ratios.append(inlier_ratio(registration["src_points"], registration["ref_points"], data["transform"], 0.1, registration["batch"]))
        rmses.append(root_mean_square_alignment_error(registration["transform"], data["transform"], data["information"]))

ratios, rmses = torch.cat(ratios), torch.cat(rmses)
print(f"IR {100 * ratios.mean():.2f}  FMR {100 * feature_matching_recall(ratios, threshold=0.05):.2f}  RR {100 * registration_recall(rmses, threshold=0.2):.2f}")

# IR 71.90  FMR 98.00  RR 98.00

!!! note "The published number averages per scene"
    3DMatch reports every number averaged within each of the 8 test scenes first, and counts the registration recall on the non-consecutive fragment pairs only. `examples/geotransformer/registration_3dmatch_benchmark.py` is this loop with those rules in place; `--benchmark 3DLoMatch` runs the low-overlap pairs. `registration_kitti_benchmark.py` scores the KITTI odometry pairs with the 5 degree / 2 m registration recall of that benchmark, and `registration_modelnet40_benchmark.py` the ModelNet40 pairs.

## Train

You own the training loop. `GeoTransformerLoss` reads the ground-truth superpoint correspondences the model builds when `forward` receives the ground-truth `transform`, so the training forward passes it; `RandomPairRotate` rotates one cloud of the pair and keeps that transform consistent.

In [ ]:
import torch_pointcloud.transforms as T
from torch_pointcloud.losses import GeoTransformerLoss

train_transform = T.Compose([
    T.RandomSample(keys=DataKeys.POS, num_samples=30000, allow_fewer=True),
    T.RandomSample(keys=DataKeys.REF_POS, num_samples=30000, allow_fewer=True),
    T.RandomPairRotate(src_keys=DataKeys.POS, ref_keys=DataKeys.REF_POS, transform_key=DataKeys.TRANSFORM),
    T.RandomJitter(keys=[DataKeys.POS, DataKeys.REF_POS], sigma=0.0025, clip=None, distribution="uniform"),
])
train_dataset = ThreeDMatch(root="data", split="train", transform=train_transform)
train_dataloader = PointCloudDataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=4,
    cat_keys=[DataKeys.REF_POS],
    stack_keys=[DataKeys.TRANSFORM],
)

model = tp.create_model("geotransformer.3dmatch.zheng-qin", task="registration").cuda()
criterion = GeoTransformerLoss(positive_radius=0.05)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-6)

num_epochs = 1  # a demo pass, raise it to actually train
model.train()
for epoch in range(num_epochs):
    total_loss = 0.0
    for data in tqdm(train_dataloader, desc=f"Epoch {epoch}"):
        data = {k: v.cuda() if torch.is_tensor(v) else v for k, v in data.items()}
        out = model(
            None,
            data["pos"],
            data["batch"],
            None,
            data["ref_pos"],
            data["batch_ref_pos"],
            transform=data["transform"],
        )
        loss = criterion(out, data)["loss"]

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Loss epoch {epoch}: {total_loss / len(train_dataloader):.3f}")

`examples/geotransformer/registration_3dmatch_train.py` is the authors' recipe: 40 epochs of Adam at $10^{-4}$, decayed by 0.95 every epoch, one pair per step.